# Lab 2: Build multi-agent workflows with Strands

In this lab, you'll advance from single-agent systems to sophisticated multi-agent architectures using Strands Agents' ["Agents as Tools"](https://strandsagents.com/latest/documentation/docs/user-guide/concepts/multi-agent/agents-as-tools/) pattern. You'll create specialized financial agents that work together under an orchestrator to provide comprehensive financial guidance.

### Modification: OpenAI-Compatible SageMaker Endpoint

This version uses **Qwen 3.5 9B on a SageMaker endpoint** (via the OpenAI-compatible API) for the Financial Analysis Agent.

**Endpoint pattern:**
```
https://runtime.sagemaker.<REGION>.amazonaws.com/endpoints/<ENDPOINT_NAME>/openai/v1
```

Authentication uses auto-refreshing bearer tokens generated from AWS credentials.

In [ ]:
%pip install --upgrade --quiet --no-warn-conflicts strands-agents strands-agents-tools yfinance pydantic openai httpx sagemaker boto3 python-dotenv

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel
from strands.models.openai import OpenAIModel
from strands.agent.conversation_manager import SummarizingConversationManager
import yfinance as yf
from typing import List
import httpx
from openai import AsyncOpenAI
from sagemaker.core.token_generator import generate_token

### Step 1: Wrap Budget Agent as a Tool

In [ ]:
import sys
sys.path.insert(0, "strands_agents")
from budget_agent import FinancialReport, budget_agent

@tool
def budget_agent_tool(query: str) -> FinancialReport:
    """Generate structured financial reports with budget analysis and recommendations."""
    try:
        structured_response = budget_agent.structured_output(output_model=FinancialReport, prompt=query)
        return structured_response
    except Exception as e:
        return FinancialReport(monthly_income=0.0, budget_categories=[], recommendations=[f"Error: {str(e)}"], financial_health_score=1)

print("✓ Budget Agent tool registered")

### Step 2: Create the Financial Analysis Agent (Qwen 3.5 9B via OpenAI-Compatible SageMaker API)

In [ ]:
FINANCIAL_ANALYSIS_PROMPT = """You are a specialized financial analysis agent focused on investment research and portfolio recommendations. Your role is to:

1. Research and analyze stock performance data
2. Create diversified investment portfolios
3. Provide data-driven investment recommendations

You do not provide specific investment advice but rather present analytical data to help users make informed decisions. Always include disclaimers about market risks and the importance of consulting financial advisors."""

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path("..") / ".env", override=True)

SAGEMAKER_ENDPOINT_NAME = os.environ["SAGEMAKER_ENDPOINT_NAME"]
SAGEMAKER_REGION = os.environ["AWS_REGION"]

base_url = f"https://runtime.sagemaker.{SAGEMAKER_REGION}.amazonaws.com/endpoints/{SAGEMAKER_ENDPOINT_NAME}/openai/v1"

class SageMakerAuth(httpx.Auth):
    """Auto-refreshing auth that generates a fresh bearer token on each request."""
    def __init__(self, region: str):
        self.region = region
    def auth_flow(self, request):
        request.headers["Authorization"] = f"Bearer {generate_token(region=self.region)}"
        yield request

async_http_client = httpx.AsyncClient(auth=SageMakerAuth(region=SAGEMAKER_REGION))
strands_client = AsyncOpenAI(base_url=base_url, api_key="sagemaker", http_client=async_http_client)

qwen_model = OpenAIModel(client=strands_client, model_id="", params={"temperature": 0.7, "max_tokens": 4096})

print(f"✓ Qwen 3.5 9B model configured via SageMaker OpenAI-compatible API")
print(f"  Endpoint: {SAGEMAKER_ENDPOINT_NAME}")
print(f"  Base URL: {base_url}")

In [ ]:
@tool
def get_stock_analysis(symbol: str) -> str:
    """Get comprehensive analysis for a specific stock symbol."""
    try:
        stock = yf.Ticker(symbol)
        info = stock.info
        hist = stock.history(period="1y")
        current_price = hist["Close"].iloc[-1]
        year_high = hist["High"].max()
        year_low = hist["Low"].min()
        avg_volume = hist["Volume"].mean()
        price_change = ((current_price - hist["Close"].iloc[0]) / hist["Close"].iloc[0]) * 100
        return f"Stock Analysis for {symbol.upper()}: Price=${current_price:.2f}, 52W High=${year_high:.2f}, 52W Low=${year_low:.2f}, YTD Change={price_change:.2f}%, Avg Volume={avg_volume:,.0f}, Company={info.get('longName', 'N/A')}, Sector={info.get('sector', 'N/A')}"
    except Exception as e:
        return f"Unable to retrieve data for {symbol}: {str(e)}"

In [ ]:
@tool
def create_diversified_portfolio(risk_level: str, investment_amount: float) -> str:
    """Create a diversified portfolio based on risk level (conservative, moderate, aggressive) and investment amount."""
    portfolios = {
        "conservative": {"stocks": ["AAPL", "MSFT", "JNJ", "PG", "KO"], "weights": [0.25, 0.25, 0.20, 0.15, 0.15], "description": "Focus on large-cap, dividend-paying stocks"},
        "moderate": {"stocks": ["AAPL", "GOOGL", "AMZN", "TSLA", "NVDA"], "weights": [0.30, 0.25, 0.20, 0.15, 0.10], "description": "Balanced mix of growth and stability"},
        "aggressive": {"stocks": ["TSLA", "NVDA", "AMZN", "GOOGL", "META"], "weights": [0.30, 0.25, 0.20, 0.15, 0.10], "description": "High-growth potential stocks"},
    }
    if risk_level.lower() not in portfolios:
        return "Risk level must be: conservative, moderate, or aggressive"
    portfolio = portfolios[risk_level.lower()]
    result = f"{risk_level.upper()} Portfolio (${investment_amount:,.0f}): {portfolio['description']}\n"
    for stock, weight in zip(portfolio["stocks"], portfolio["weights"]):
        result += f"  {stock}: {weight*100:.0f}% (${investment_amount*weight:,.0f})\n"
    result += "\n⚠️ Disclaimer: This is for educational purposes only."
    return result

In [ ]:
@tool
def compare_stock_performance(symbols: List[str], period: str = "1y") -> str:
    """Compare performance of multiple stocks over a specified period (1y, 6m, 3m, 1m)."""
    if len(symbols) > 5:
        return "Please limit comparison to 5 stocks maximum"
    try:
        performance_data = {}
        for symbol in symbols:
            hist = yf.Ticker(symbol).history(period=period)
            if not hist.empty:
                performance = ((hist["Close"].iloc[-1] - hist["Close"].iloc[0]) / hist["Close"].iloc[0]) * 100
                performance_data[symbol] = performance
        result = f"Stock Performance Comparison ({period}):\n"
        for stock, performance in sorted(performance_data.items(), key=lambda x: x[1], reverse=True):
            result += f"  {stock}: {performance:+.2f}%\n"
        return result
    except Exception as e:
        return f"Error comparing stocks: {str(e)}"

In [ ]:
financial_analysis_agent = Agent(
    model=qwen_model,
    system_prompt=FINANCIAL_ANALYSIS_PROMPT,
    tools=[get_stock_analysis, create_diversified_portfolio, compare_stock_performance],
)
print("✓ Financial Analysis Agent created (Qwen 3.5 9B — SageMaker OpenAI-compatible API)")

In [ ]:
# Test financial analysis agent
response = financial_analysis_agent("Create a moderate risk portfolio for $10,000 and analyze Apple stock")

### Step 3: Wrap Financial Analysis Agent as a Tool

In [ ]:
@tool
def financial_analysis_agent_tool(query: str) -> str:
    """Handle investment analysis queries including stock research, portfolio creation, and performance comparisons."""
    try:
        response = financial_analysis_agent(query)
        return str(response)
    except Exception as e:
        return f"Financial analysis error: {str(e)}"

print("✓ Financial Analysis Agent tool registered")

### Step 4: Create the Orchestrator Agent

The orchestrator uses **Claude Haiku 4.5** via Amazon Bedrock.

In [ ]:
ORCHESTRATOR_PROMPT = """You are a comprehensive financial advisor orchestrator that coordinates between specialized financial agents.

Your specialized agents are:
1. **budget_agent_tool**: Handles budgeting, spending analysis, savings recommendations
2. **financial_analysis_agent_tool**: Handles investment analysis, stock research, portfolio creation

Guidelines:
- Use **budget_agent_tool** for: budgets, spending, expenses, savings, debt
- Use **financial_analysis_agent_tool** for: stocks, investments, portfolios, market analysis
- You can use both agents together for comprehensive financial planning
- Always provide a cohesive summary with actionable next steps"""

In [ ]:
orchestrator_model = BedrockModel(
    model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    region_name="us-west-2",
    temperature=0.0,
)

In [ ]:
conversation_manager = SummarizingConversationManager(summary_ratio=0.3, preserve_recent_messages=5)

In [ ]:
orchestrator_agent = Agent(
    model=orchestrator_model,
    system_prompt=ORCHESTRATOR_PROMPT,
    tools=[budget_agent_tool, financial_analysis_agent_tool],
    conversation_manager=conversation_manager,
)
print("✓ Orchestrator Agent created (Claude Haiku 4.5 — Bedrock)")

### Step 5: Test the Multi-Agent System

In [ ]:
# Test orchestrator with complex multi-agent query
response = orchestrator_agent(
    "Compare Tesla and Apple stocks, and tell me if I can afford to invest $2000 with my $4000 monthly income."
)

In [ ]:
# Test budget-focused query
response = orchestrator_agent(
    "I make $6000/month and want to start investing $500/month. Help me create a budget and suggest an investment portfolio."
)